# 2stage-CPLS compression of the US east-coast AIS

Run this notebook live (interactive figures) on Binder: [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/smehri/CATraj/main?labpath=notebooks%2F03_two_stage_cpls_compression.ipynb)
Or view it on [nbviewer](https://nbviewer.org/github/smehri/CATraj/blob/main/notebooks/03_two_stage_cpls_compression.ipynb). The figures are interactive (hvplot / bokeh: zoom, pan, hover, click a legend entry), so they do not show on github.com, whose notebook viewer does not run JavaScript.

Trajectory compression keeps the points of a track that matter and drops the
rest. This notebook compresses **all** the clean AIS of the US east coast (November and December 2017) with
**2stage-CPLS**, the two-stage topology-preserving method of

> Mehri, S., Alesheikh, A. A., Basiri, A. (2021). *A Contextual Hybrid Model for Vessel Movement Prediction.* IEEE Access 9, 45600–45613. [doi:10.1109/ACCESS.2021.3066463](https://doi.org/10.1109/ACCESS.2021.3066463)

with the NOAA shoreline as the barrier, and saves the result for the prediction notebook
(`05_clstm_prediction.ipynb`), which trains the CLSTM networks on it.

1. **The method**: CPLS and 2stage-CPLS, what each stage does, and what a barrier made of *lines* means
2. **Load** the shoreline and look at the AIS files
3. **Trips**: the clean tracks are split into trips at gaps; tiny trips are dropped
4. **Compress**: `two_stage_cpls_compress` over every trip, with the shoreline as barrier
5. **Checks**: no compressed segment crosses the shoreline where the original did not
6. **Run month by month** and save: `compressed_ais_<month>.parquet`, `trips_ais_<month>.parquet`, `summary.csv`
7. **Results**: compression ratio overall, by vessel group and by vessel type; vessels before and after on a map; the stays
8. **Checks, timings and files**

**Inputs**: `AIS/ais_<month>.parquet` and `Coastline/shoreline.parquet` of the preprocessed
folder of the US data (written by `01_preprocessing.ipynb` with `DATASET = "us"`). **Outputs**: `results/03_two_stage_cpls_compression/<DATA>/`.
The package is built on [MovingPandas](https://movingpandas.org):

> Graser, A. (2019). *MovingPandas: Efficient Structures for Movement Data in Python.* GI_Forum 2019, 7(1), 54–68. [doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54)

## Data

This notebook uses the **US data** (`data/UsAIS`), the data of the 2021 paper (its section III-B):
the AIS broadcasts of the US Coast Guard that [MarineCadastre.gov](https://marinecadastre.gov/ais/) publishes
(one file per day; the 2017 files are used), and the **NOAA Medium Resolution Shoreline** as the barrier of the compression.

* **Study area**: the east coast, longitude -82 to -60 and latitude 23 to 46, **minus** the Gulf side of Florida
  and **minus** the Great Lakes (Florida to Maine). The paper's own polygon is not published; this one holds
  about twice as many positions as the paper reports for its "eastern coast".
* **Period**: 1 November to 31 December 2017, preprocessed **one month at a time** (`ais_2017-11.parquet`,
  `ais_2017-12.parquet`) with the same steps as the Zenodo archive (notebook 01): ship MMSI, valid ranges, duplicates
  and position jumps removed, a 30 s median filter, projection to UTM zone 18N (EPSG:32618, metres), vessel type and the
  paper's vessel groups. The reports are already thinned to about one per minute (median interval 70 s).
* **Shoreline**: the NOAA shoreline **lines** (`shoreline.parquet`), clipped to the study area, in the same CRS (24,073 lines on the full data).
* **Columns**: `mmsi`, `unix_time`, `x`, `y`, `lon`, `lat`, `speed_over_ground` (knots), `course_over_ground`,
  `course_over_ground_grid`, `true_heading`, `true_heading_grid`, `navigational_status`, `vessel_type`, `vessel_group`, `length`, `width`.

**Why this data.** 2stage-CPLS and the CLSTM model were developed and evaluated on it in the paper, so the compression
(this notebook) and the prediction (notebook 05) use it. The Brest data (Zenodo archive) is used in notebooks 02 (annotation with weather and sea state)
and 04 (CADP compression).

By default the notebook runs on the **committed, preprocessed two-day sample**: 2017-11-01 and 2017-11-02, the
New York / New Jersey harbour (465,795 positions, 507 vessels) and the shoreline clipped to it. To process the **full data**
(about 115 million positions), run `notebooks/01_preprocessing.ipynb` first with `DATASET = "us"` (its download and preprocessing sections),
then set `DATA = "full"` below, or run the notebook with the environment variable `CATRAJ_DATA=full`.
The full run goes month by month and, inside a month, through blocks of vessels (see "Parallel processing"), so it does not need the whole data in memory.

In [1]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")   # see catraj/__init__.py

DATA = os.environ.get("CATRAJ_DATA", "sample")      # "sample" or "full"
N_JOBS = int(os.environ.get("CATRAJ_N_JOBS", "-1"))  # -1: every core; 1: serial
MAX_DAYS = int(os.environ.get("CATRAJ_MAX_DAYS", "0"))   # 0: all the data; k: only the first k days (timing trials)

import threading
import time
from pathlib import Path

import geopandas as gpd
import geoviews as gv
import holoviews as hv
import hvplot.pandas  # noqa: F401  -- registers .hvplot
import numpy as np
import pandas as pd
import psutil
import pyarrow as pa
import pyarrow.parquet as pq
import shapely
from pyproj import Transformer

hv.extension("bokeh")                  # interactive figures: zoom, pan, hover, click a legend entry

import catraj                          # first: sets up the environment (DLL paths, threading)
from catraj import parallel_map
from catraj.compression import (Barriers, compression_ratio, segment_crossings,
                                two_stage_cpls_compress, two_stage_cpls_select)
from catraj.datasets import full_root, repo_root, sample_root
from catraj.preprocessing import get_dataset, split_trips

if DATA not in ("sample", "full"):
    raise ValueError(f"DATA must be 'sample' or 'full', not {DATA!r}")
SPEC = get_dataset("us")                 # CRS, area, period and files of the US data
CRS = SPEC.crs                           # 32618: UTM zone 18N, metres
ROOT = repo_root()
PRE = sample_root("us") if DATA == "sample" else full_root("us")
AIS_FILES = SPEC.ais_outputs(PRE)        # one Parquet file per month
if MAX_DAYS:
    AIS_FILES = AIS_FILES[:1]
SHORELINE_FILE = PRE / "Coastline" / "shoreline.parquet"
OUT = ROOT / "results" / "03_two_stage_cpls_compression" / (DATA + (f"_{MAX_DAYS}d" if MAX_DAYS else ""))
OUT.mkdir(parents=True, exist_ok=True)

# ---- the parameters of the method (the paper's, section V-A) --------------------
EPSILON = 1_000.0        # Douglas-Peucker threshold, metres
SPEED_EPSILON = 1.54     # speed threshold, metres per second (3 knots)
STAY_RADIUS = 100.0      # a movement stay: within 100 m ...
STAY_DURATION = 600.0    # ... for at least 600 s
STATUS_COLUMN = "navigational_status"   # codes 1 (at anchor) and 5 (moored) are stays
# ---- the trips --------------------------------------------------------------------
MAX_GAP = 1_800.0        # a gap of over 30 minutes between two reports starts a new trip
MIN_TRIP_POINTS = 10     # trips with fewer positions are dropped
# ---- memory: vessels are processed in blocks of about this many positions -----------
CHUNK_ROWS = 4_000_000
# ---- vessel groups left out: the paper drops the military group (20 trajectories in all, too few to learn from) ---
DROP_GROUPS = ("Military",)
# ---- the checks (section 5): the share of trips checked (1: all of them) ---------------
CHECK_RAW = 1.0 if DATA == "sample" else 0.05          # new crossings against the raw track
CHECK_COLLAPSED = 1.0 if DATA == "sample" else 0.01    # recomputed selection against the collapsed track

print(f"DATA = {DATA!r}, N_JOBS = {N_JOBS}, MAX_DAYS = {MAX_DAYS or 'all'}\ninput:  {PRE}\noutput: {OUT}")
print("AIS files:", [f.name for f in AIS_FILES])

DATA = 'full', N_JOBS = -1, MAX_DAYS = all
input:  C:\dev\CATraj\data\UsAIS\preprocessed
output: C:\dev\CATraj\results\03_two_stage_cpls_compression\full
AIS files: ['ais_2017-11.parquet', 'ais_2017-12.parquet']


## Parallel processing

Three steps of this notebook run in parallel, all through the package's `n_jobs`
(`catraj.parallel_map` / `map_groups`, `src/catraj/_parallel.py`):

* **`split_trips(..., n_jobs=N_JOBS)`**: every vessel's track is split independently of
  the others, so the vessels are shared out over the processes.
* **`two_stage_cpls_compress(..., by="trip_id", n_jobs=N_JOBS)`**: every trip is
  compressed independently of the others (the speed series, the stays and the topological
  checks only ever look at one trip), so the trips are shared out. They are grouped
  into chunks of about equal number of positions, because the cost depends on a trip's length.
* **The checks** (`catraj.parallel_map` over the trips): the same, one task per trip.

`N_JOBS` follows the scikit-learn convention: `1` runs everything serially in this
process, `-1` uses every core, `k` uses `k` processes. The results come back in task order and
every trip is computed by the same code either way, so **parallel and serial runs give
identical results**; only the wall time changes. To run serially, set `N_JOBS = 1` or start Jupyter with `CATRAJ_N_JOBS=1`.

**Memory on the full data.** One month is about 60 million positions and the whole data about 115 million: too much to hold as
trips, geometries and compressed copies at once. The notebook therefore reads each month's file in its row groups, which
are sorted by vessel, and takes **blocks of about `CHUNK_ROWS` positions that never split a vessel** (a
vessel's rows stay together, so its trips are never cut by the block boundary). A block is split into trips, compressed, checked and
appended to the month's output files, then dropped. Each worker receives its share of the trips (pickled) and the shoreline
(a few tens of MB on the full data), so the memory in use is roughly one block in this process plus the chunks in flight in the workers.
The peak of this process and its workers is measured and reported at the end.

## 1 · The method

**CPLS** (Context-aware Piecewise Linear Segmentation, Algorithm 1 of the paper) is
Douglas–Peucker with a memory of the world. Douglas–Peucker (DP) drops a point when it lies
within ε of the line between its neighbours: a statement about *distance* only. On a
coastal route it will replace a bend around a headland by a chord over the land, or move an
island from one side of the track to the other. CPLS runs DP and then, **inside every
segment that DP collapsed**, removes the points one by one, front to back, and puts a
point back whenever removing it would change the track's *topological signature*
(after Tienaah et al.): the relation of the simplified track with each neighbouring
barrier (DE-9IM: no crossing of the shoreline that the original did not cross), the side on which each
barrier lies (a nine-cell direction model), and how the track meets itself (no new
self-intersection, none lost, no fold-back of two consecutive segments). The result is always a
superset of the DP result and never introduces a new crossing of the shoreline.
`help(catraj.compression.cpls)` lists the decisions behind it.

**Why a second stage.** Plain CPLS on raw AIS keeps most points of a vessel at a berth:
the position of a moored vessel jitters by a few metres, its track crosses itself thousands
of times, and every crossing is part of the signature. Worse, simplified away, "it appears
as if the vessel moves slowly, whereas in the original uncompressed trajectory the vessel
stops moving for a period of time" (de Vries and van Someren, 2012).
**2stage-CPLS** (Algorithm 2 of the paper) therefore works in stages:

1. **Stays.** A *stay point* is a position whose AIS status says at anchor or moored (codes
   1 and 5), or that belongs to a *movement stay* (Li et al.'s stay-point detection: the
   vessel remains within `STAY_RADIUS` of a spot for at least `STAY_DURATION`). Each stay is
   replaced by two vertices at its centroid, the arrival and the departure, with their own
   times: the jitter is gone, the duration of the stop is kept.
2. **Speed.** Douglas–Peucker on the *speed series* (speed from positions and time, with error
   `|v - v'|`, `v'` interpolated in time) keeps the points where the speed changes by more
   than `SPEED_EPSILON`: a vessel that brakes, accelerates or stops keeps the points that say so.
3. **Position.** Between those forced vertices, Douglas–Peucker with `EPSILON` metres, and
   then the topological pass of CPLS over the collapsed track.

The compressed track is therefore the stay-collapsed track, simplified; the
guarantee about the shoreline is relative to that collapsed track (see the checks).
`help(catraj.compression.two_stage)` lists every decision, with the sentence of the paper behind it.

**What we use.** The paper's parameters: `EPSILON = 1,000 m`, `SPEED_EPSILON = 1.54 m/s` (3 knots), stay radius 100 m, stay
duration 600 s and the status column `navigational_status`. All five CPLS relations stay on.

**The barrier is a set of lines.** The NOAA shoreline holds *lines*, not land polygons (the Brest notebooks use polygons).
`Barriers` accepts both. With a polygon, a segment can enter the land, lie in it or leave it; with a shoreline line the only thing
that can happen is that a segment **meets the line**, and a segment of a vessel's track that meets the shoreline line **crosses the coastline**
(goes from water to land or the other way round) or, at best, touches it. So "no new crossing of the barrier" means here
*no new crossing of the coastline*: the compressed track does not run over a headland, a spit or an island that the original sailed around.
The line does not say which side is land, so the side relations of the signature compare which side of the *line* the track passes on, as they do for
a polygon's boundary. Two consequences to keep in mind when reading the checks: the NOAA line is generalised (about 1:70,000), so original tracks in
harbours, rivers and along piers cross it often, and the guarantee is always **relative to the original** (a crossing counts only if the replaced stretch of
the original did not cross); and a vessel moored at a quay is within metres of the line, so moving its stay to the centroid can make a segment touch the line.

**Filtering.** `two_stage_cpls_compress` median-filters the positions over 30 s by default
(and the speed series with the same window). The preprocessed AIS **already is** filtered
(notebook 01 applies the same 30 s median), so filtering again would only smooth smooth
data, change the positions the CLSTM later trains on and cost time. We pass
`filter_window=None`: the compressed rows are then exactly rows of the clean data, with
their original values, except that a stay's two kept points move to the stay's centroid.

## 2 · Load the shoreline and the AIS files

The shoreline is a `GeoDataFrame` of lines in EPSG:32618 (UTM zone 18N, metres, as the distance thresholds
need). `Barriers(...)` indexes the lines once so that every trip reuses the index. The AIS is **not** loaded as a whole:
each month is a Parquet file with the rows sorted by `(mmsi, unix_time)` and several row groups, read in blocks of vessels in section 6.
Here only the files' sizes are looked at.

In [2]:
t0 = time.perf_counter()
coast = gpd.read_parquet(SHORELINE_FILE)
barriers = Barriers(coast)
LOAD_S = time.perf_counter() - t0
meta = [pq.ParquetFile(f).metadata for f in AIS_FILES]
print(f"{len(coast):,} shoreline lines ({shapely.get_num_coordinates(coast.geometry.values).sum():,} vertices), "
      f"{coast.geom_type.unique().tolist()}, crs {coast.crs.to_string()}; barriers built in {LOAD_S:.1f} s")
for f, m in zip(AIS_FILES, meta):
    print(f"{f.name}: {m.num_rows:,} positions in {m.num_row_groups} row group(s), {f.stat().st_size / 1e6:,.0f} MB")
pq.ParquetFile(AIS_FILES[0]).schema_arrow.to_string().replace("\n", " | ")[:900]

24,073 shoreline lines (1,117,966 vertices), ['LineString'], crs EPSG:32618; barriers built in 0.1 s
ais_2017-11.parquet: 59,759,561 positions in 79 row group(s), 1,316 MB
ais_2017-12.parquet: 55,272,838 positions in 73 row group(s), 1,191 MB


'mmsi: int64 | unix_time: int64 | x: double | y: double | lon: double | lat: double | speed_over_ground: float | course_over_ground: float | true_heading: float | course_over_ground_grid: float | true_heading_grid: float | navigational_status: float | ship_type_code: float | vessel_type: dictionary<values=string, indices=int8, ordered=0> | vessel_group: dictionary<values=string, indices=int8, ordered=0> | length: float | width: float | -- schema metadata -- | pandas: \'{"index_columns": [], "column_indexes": [], "columns": [{"name":\' + 2225'

## 3 · Trips

A vessel's AIS history is one long track: over days or months it is many voyages joined
end to end, with periods out of range in between. Compression and, later, the prediction
network work on **trips**, so `catraj.preprocessing.split_trips` cuts each vessel's track
wherever two consecutive reports are **more than `MAX_GAP` = 30 minutes** apart: the
vessel was out of range or its transponder was off, so the positions on either side are
not one movement and a chord across the gap would invent a path. 30 minutes is far longer than
the reporting interval of this data (about a minute) and short enough not to join separate voyages.

Stays are *not* used to cut the trips here: a stay is part of a trip (the vessel waits and
leaves again), and finding and collapsing stays is exactly what stage 1 of 2stage-CPLS does
inside each trip. So `split_trips` is called with no stay rule (`stay_radius=None`).

Some pieces are tiny: a handful of reports before the vessel goes out of range, a single
fix between two long gaps. They carry no trajectory a model could learn from, so trips with fewer than
`MIN_TRIP_POINTS` = 10 positions are **dropped** from both the uncompressed and the
compressed data. Every trip gets a `trip_id`, **unique over the whole dataset** (the two months are
processed separately, and the ids of the second month continue after the first's), which
both output files carry. The uncompressed, trip-split data is saved as `trips_ais.parquet`:
notebook 05 trains on it too, so the trip definition is the same for both models. The months are
cut at their boundary (a trip that spans midnight of 30 November has two ids), since the data
is preprocessed per month.

`make_trips(block, first_id)` does this for one block of vessels.

In [3]:
def make_trips(block, first_id):
    """Split a block of vessels (a DataFrame sorted by mmsi, unix_time) into trips.

    Returns the trips as a GeoDataFrame with a fresh index and the column ``trip_id``
    (``first_id``, ``first_id + 1``, ...), and the counts (trips before dropping, tiny trips, tiny positions).
    """
    g = gpd.GeoDataFrame(block, geometry=gpd.points_from_xy(block.x, block.y), crs=CRS)
    g = split_trips(g, by="mmsi", time_column="unix_time", max_gap=MAX_GAP,
                    stay_radius=None, stay_duration=None, stay_column=None,
                    trip_column="trip", n_jobs=N_JOBS)
    sizes = g.groupby(["mmsi", "trip"], sort=False).size()
    small = sizes < MIN_TRIP_POINTS
    counts = dict(trips_all=len(sizes), tiny=int(small.sum()), tiny_points=int(sizes[small].sum()))
    big = sizes[~small].index.to_frame(index=False)
    trips = g.merge(big, on=["mmsi", "trip"])
    trips = trips.sort_values(["mmsi", "unix_time"], kind="stable").reset_index(drop=True)
    trips["trip_id"] = trips.groupby(["mmsi", "trip"], sort=False).ngroup().astype("int64") + first_id
    return trips.drop(columns="trip"), counts

## 4 · Compress

`two_stage_cpls_compress(traj, epsilon, speed_epsilon, barriers, ...)` takes the points of
all trips as one `GeoDataFrame`, with `by="trip_id"` compressing every trip on its own, and
returns only the retained rows, every column intact (index label, time, speed, course, status,
vessel group and type). Its `stay` column is True on the two kept points of every stay.

All the trips of a block are compressed in one call (the trips are shared out over `N_JOBS` processes; every call sends the shoreline to the workers, which
costs a few seconds on the full shoreline, so the blocks are not split further, by vessel group for instance).
Seconds are wall time. The compressed rows keep the geometry of the retained points, in which a stay's two ends are
at the stay's centroid, so `x`, `y`, `lon`, `lat` are rewritten from that geometry so that they agree with it (for every other row they are unchanged).

In [4]:
PARAMS = dict(time_column="unix_time", by="trip_id", status_column=STATUS_COLUMN,
              stay_radius=STAY_RADIUS, stay_duration=STAY_DURATION, stay_column="stay",
              filter_window=None, n_jobs=N_JOBS)       # the data is already filtered (section 1)
to_ll = Transformer.from_crs(CRS, 4326, always_xy=True)


def compress_trips(trips):
    """2stage-CPLS over every trip of the block.

    Returns the retained rows (index = the trips' row numbers, so input order), a stay's points at its centroid.
    """
    kept = two_stage_cpls_compress(trips, EPSILON, SPEED_EPSILON, barriers, **PARAMS).copy()
    gx, gy = kept.geometry.x.to_numpy(), kept.geometry.y.to_numpy()
    moved = (gx != kept.x.to_numpy()) | (gy != kept.y.to_numpy())      # the stay points, now at their centroid
    kept.loc[moved, "lon"], kept.loc[moved, "lat"] = to_ll.transform(gx[moved], gy[moved])
    kept["x"], kept["y"] = gx, gy
    return kept

## 5 · Checks

**No new crossing of the shoreline.** The package defines a *new crossing* as a segment of the
compressed track that touches a barrier although the stretch of the original track it replaces does not
(`segment_crossings(x, y, barriers, keep=mask, new_only=True)`); with the shoreline as the barrier, that is a segment that
crosses (or touches) the coastline where the original did not. Two measurements, made on every block while it is in memory
(all the trips on the sample; a random share of them on the full data, set by `CHECK_RAW` and `CHECK_COLLAPSED`):

* **Against the raw track**, for a share of the trips: the clean positions and the retained
  ones. This is not what 2stage-CPLS guarantees, because a stay's points are moved to their
  centroid, up to the extent of the stay away from where the vessel was, so a segment that starts or ends
  at a moved stay point can touch the shoreline where the raw one did not (a vessel at a quay is within metres
  of the line). The count says how many of the new crossings, if any, are at a stay point.
* **Against the collapsed track**, the reference the algorithm keeps: `two_stage_cpls_select` is run again
  on a random share of the trips, and its own
  collapsed coordinates and mask are measured. The guarantee says this count is **zero**; the check
  also compares the recomputed mask with the rows kept.

Both run in parallel over the trips (one call per block). `check_block` returns the counts of one block.

In [5]:
def check_trip(args):
    """Both checks on one trip. Returns (new crossings against the raw track, of them at a stay point,
    new crossings against the collapsed track, 1 if the recomputed mask is the saved one); -1 where a check was not asked for."""
    x, y, t, status, keep, stay, do_raw, do_col = args
    raw_new = raw_stay = col_new = col_same = -1
    if do_raw:
        seg = segment_crossings(x, y, barriers, keep=keep, new_only=True)
        raw_new = int(seg.sum())
        raw_stay = 0
        if raw_new:
            kept_i = np.flatnonzero(keep)
            raw_stay = int((seg & (stay[kept_i[:-1]] | stay[kept_i[1:]])).sum())
    if do_col:
        sel = two_stage_cpls_select(x, y, t, EPSILON, SPEED_EPSILON, barriers, status=status,
                                    stay_radius=STAY_RADIUS, stay_duration=STAY_DURATION, speed_window=None)
        col_new = int(segment_crossings(sel.x, sel.y, barriers, keep=sel.mask, new_only=True).sum())
        col_same = int((sel.mask == keep).all())
    return raw_new, raw_stay, col_new, col_same


def check_block(trips, kept, rng):
    """Both checks on a random share of the block's trips (one parallel call); returns a dict of counts."""
    n = len(trips)
    keep = np.zeros(n, dtype=bool)
    keep[kept.index.to_numpy()] = True
    stay = np.zeros(n, dtype=bool)
    stay[kept.index[kept.stay.to_numpy()].to_numpy()] = True
    xs, ys = trips.x.to_numpy(), trips.y.to_numpy()
    t_all = trips.unix_time.to_numpy(dtype=float)
    st_all = trips[STATUS_COLUMN].to_numpy(dtype=float)
    bounds = np.flatnonzero(np.diff(trips.trip_id.to_numpy())) + 1
    starts, ends = np.r_[0, bounds], np.r_[bounds, n]
    do_raw, do_col = rng.random(len(starts)) < CHECK_RAW, rng.random(len(starts)) < CHECK_COLLAPSED
    idx = np.flatnonzero(do_raw | do_col)
    tasks = [(xs[starts[i]:ends[i]], ys[starts[i]:ends[i]], t_all[starts[i]:ends[i]], st_all[starts[i]:ends[i]],
              keep[starts[i]:ends[i]], stay[starts[i]:ends[i]], bool(do_raw[i]), bool(do_col[i])) for i in idx]
    res = np.array(parallel_map(check_trip, tasks, N_JOBS, sizes=(ends - starts)[idx].tolist())).reshape(-1, 4)
    r, c = res[res[:, 0] >= 0], res[res[:, 2] >= 0]
    return dict(raw_trips=len(r), raw_new=int(r[:, 0].sum()), raw_new_at_stay=int(r[:, 1].sum()),
                raw_trips_with_new=int((r[:, 0] > 0).sum()),
                col_trips=len(c), col_new=int(c[:, 2].sum()), col_same=int(c[:, 3].sum()))

## 6 · Run month by month and save

Each month's file is read in its row groups (sorted by vessel); blocks of about `CHUNK_ROWS` positions that end at a vessel boundary
go through `make_trips`, `compress_trips` and `check_block`, and are appended to the month's two Parquet files, so the memory stays flat.
The paper's **military group** (20 trajectories in all) is left out (`DROP_GROUPS`), the count is printed.
Written under `results/03_two_stage_cpls_compression/<DATA>/`:

* `compressed_ais_<month>.parquet`: the compressed points with the columns the CLSTM needs
  (`mmsi`, `trip_id`, `unix_time`, `x`, `y`, `lon`, `lat`, `speed_over_ground`, `course_over_ground`, `course_over_ground_grid`,
  `true_heading`, `true_heading_grid`, `vessel_type`, `vessel_group`, `navigational_status`, `length`, `width`) plus `stay`;
* `trips_ais_<month>.parquet`: the same columns (without `stay`) of the **uncompressed** clean AIS,
  split into trips in the same way (same `trip_id`);
* `summary.csv`: points in and out, compression ratio and seconds, overall, per month, per vessel group and per vessel type.

The layout mirrors the preprocessed folder (one file per month, as `AIS/ais_2017-11.parquet`); read all months with
`pd.read_parquet(sorted(OUT.glob("compressed_ais_*.parquet")))`. `trip_id` is unique across the months. The memory use of this process and
its workers (resident memory, summed; shared pages are counted once per process, so it is an upper bound) is sampled every second while the cell runs.

In [6]:
class PeakMemory:
    """Peak resident memory of this process and its children, sampled in a thread."""

    def __init__(self, every=1.0):
        self.every, self.peak, self._stop = every, 0, threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _sample(self):
        me = psutil.Process()
        total = me.memory_info().rss
        for child in me.children(recursive=True):
            try:
                total += child.memory_info().rss
            except psutil.Error:
                pass
        self.peak = max(self.peak, total)

    def _run(self):
        while not self._stop.is_set():
            self._sample()
            self._stop.wait(self.every)

    def __enter__(self):
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join()
        self._sample()


READ_COLUMNS = ["mmsi", "unix_time", "x", "y", "lon", "lat", "speed_over_ground", "course_over_ground",
                "course_over_ground_grid", "true_heading", "true_heading_grid", "vessel_type", "vessel_group",
                "navigational_status", "length", "width"]
OUT_COLUMNS = ["mmsi", "trip_id"] + [c for c in READ_COLUMNS if c != "mmsi"]


DROPPED = dict(positions=0, vessels=set())


def vessel_blocks(path, rows):
    """Blocks of at least ``rows`` positions that end where a vessel ends (the file is sorted by mmsi)."""
    start = pd.Timestamp(path.stem.split("_")[-1] + "-01", tz="UTC").timestamp()
    pf = pq.ParquetFile(path)
    carry, n_carry = [], 0
    for i in range(pf.num_row_groups):
        df = pf.read_row_group(i, columns=READ_COLUMNS).to_pandas()
        for c in ("vessel_type", "vessel_group"):
            df[c] = df[c].astype(object).fillna("Unknown")
        if MAX_DAYS:
            df = df[df.unix_time < start + MAX_DAYS * 86_400]
        dropped = df.vessel_group.isin(DROP_GROUPS)
        DROPPED["positions"] += int(dropped.sum())
        DROPPED["vessels"] |= set(df.mmsi[dropped].unique())
        df = df[~dropped]
        if not len(df):
            continue
        carry.append(df)
        n_carry += len(df)
        if n_carry >= rows:
            df = pd.concat(carry, ignore_index=True)
            assert df.mmsi.is_monotonic_increasing, "the file must be sorted by mmsi"
            cut = int(df.mmsi.searchsorted(df.mmsi.iat[-1], side="left"))   # the last vessel may go on in the next group
            if cut == 0:
                carry, n_carry = [df], len(df)
                continue
            yield df.iloc[:cut]
            carry = [df.iloc[cut:]]
            n_carry = len(df) - cut
    if n_carry:
        yield pd.concat(carry, ignore_index=True)


class Appender:
    """Append DataFrames to one Parquet file, with the schema of the first one."""

    def __init__(self, path):
        self.path, self.writer = path, None

    def write(self, df):
        table = pa.Table.from_pandas(pd.DataFrame(df), preserve_index=False)
        if self.writer is None:
            self.writer = pq.ParquetWriter(self.path, table.schema, compression="zstd")
        self.writer.write_table(table.cast(self.writer.schema))

    def close(self):
        if self.writer is not None:
            self.writer.close()


rng = np.random.default_rng(0)
tabs, vessel_keys, vessel_info, month_rows, check_totals = [], [], [], [], {}
N_ALL = N_TRIPS_ALL = N_TINY = N_TINY_POINTS = 0
READ_S = SPLIT_S = COMPRESS_S = CHECK_S = WRITE_S = 0.0
trip_offset = 0
for old in list(OUT.glob("compressed_ais_*.parquet")) + list(OUT.glob("trips_ais_*.parquet")):
    old.unlink()                                  # a rerun replaces the outputs

T_RUN = time.perf_counter()
with PeakMemory() as mem:
    for path in AIS_FILES:
        month = path.stem.split("_")[-1]
        w_trips, w_comp = Appender(OUT / f"trips_ais_{month}.parquet"), Appender(OUT / f"compressed_ais_{month}.parquet")
        m_in = m_out = m_trips = m_stay = 0
        m_vessels, m_t0 = set(), time.perf_counter()
        blocks = vessel_blocks(path, CHUNK_ROWS)
        while True:
            t0 = time.perf_counter()
            block = next(blocks, None)
            READ_S += time.perf_counter() - t0
            if block is None:
                break
            N_ALL += len(block)
            t0 = time.perf_counter()
            trips, counts = make_trips(block, trip_offset)
            SPLIT_S += time.perf_counter() - t0
            N_TRIPS_ALL += counts["trips_all"]; N_TINY += counts["tiny"]; N_TINY_POINTS += counts["tiny_points"]
            del block
            if not len(trips):
                continue
            n_trips = trips.trip_id.nunique()
            trip_offset += n_trips
            t0 = time.perf_counter()
            kept = compress_trips(trips)
            COMPRESS_S += time.perf_counter() - t0
            t0 = time.perf_counter()
            res = check_block(trips, kept, rng)
            CHECK_S += time.perf_counter() - t0
            for k, v in res.items():
                check_totals[k] = check_totals.get(k, 0) + v
            t0 = time.perf_counter()
            w_trips.write(trips[OUT_COLUMNS])
            w_comp.write(kept[OUT_COLUMNS + ["stay"]])
            WRITE_S += time.perf_counter() - t0
            # the bookkeeping for the summary and the figures
            keys = ["vessel_group", "vessel_type"]
            tab = trips.groupby(keys).agg(trips=("trip_id", "nunique"), points_in=("x", "size")).join(
                kept.groupby(keys).agg(points_out=("x", "size"), stay_points_out=("stay", "sum"))).fillna(0)
            tabs.append(tab)
            vessel_keys.append(trips[keys + ["mmsi"]].drop_duplicates())
            info = trips.groupby("mmsi").agg(n=("x", "size"), lon0=("lon", "min"), lon1=("lon", "max"),
                                             lat0=("lat", "min"), lat1=("lat", "max"),
                                             vessel_group=("vessel_group", "first"), vessel_type=("vessel_type", "first"))
            info["stays"] = kept.groupby("mmsi").stay.sum().reindex(info.index).fillna(0).astype(int)
            vessel_info.append(info.reset_index())
            m_in += len(trips); m_out += len(kept); m_trips += n_trips; m_stay += int(kept.stay.sum())
            m_vessels |= set(trips.mmsi.unique())
            print(f"{month}: block of {len(trips):>9,} positions, {n_trips:>7,} trips -> {len(kept):>8,} points "
                  f"(R = {compression_ratio(len(kept), len(trips)):.1f} %), "
                  f"{time.perf_counter() - T_RUN:7.0f} s elapsed, memory {mem.peak / 2**30:.1f} GiB so far", flush=True)
            del trips, kept
        w_trips.close(); w_comp.close()
        month_rows.append(dict(level="month", name=month, vessels=len(m_vessels), trips=m_trips, points_in=m_in,
                               points_out=m_out, stay_points_out=m_stay, seconds=time.perf_counter() - m_t0))
RUN_S = time.perf_counter() - T_RUN
PEAK_GIB = mem.peak / 2**30
N = sum(r["points_in"] for r in month_rows)
N_TRIPS = trip_offset
print(f"\n{N_ALL:,} positions read (military group left out), split into {N_TRIPS_ALL:,} trips at gaps of over {MAX_GAP / 60:.0f} min; "
      f"dropped {N_TINY:,} tiny trips of fewer than {MIN_TRIP_POINTS} positions "
      f"({N_TINY_POINTS:,} positions, {N_TINY_POINTS / N_ALL:.2%} of the data)")
print(f"dropped the groups {DROP_GROUPS}: {DROPPED['positions']:,} positions of {len(DROPPED['vessels']):,} vessels (as in the paper)")
print(f"kept {N_TRIPS:,} trips, {N:,} positions")
print(f"compressed to {sum(r['points_out'] for r in month_rows):,} points; whole run {RUN_S:.0f} s on {N_JOBS} job(s); "
      f"peak memory of this process and its workers {PEAK_GIB:.1f} GiB")

2017-11: block of 4,022,758 positions,   5,318 trips ->  127,311 points (R = 96.8 %),     102 s elapsed, memory 8.9 GiB so far


2017-11: block of 4,518,550 positions,   5,547 trips ->  102,679 points (R = 97.7 %),     162 s elapsed, memory 9.8 GiB so far


2017-11: block of 4,498,767 positions,   9,750 trips ->  218,667 points (R = 95.1 %),     237 s elapsed, memory 9.9 GiB so far


2017-11: block of 4,497,494 positions,   5,382 trips ->  220,465 points (R = 95.1 %),     323 s elapsed, memory 9.9 GiB so far


2017-11: block of 4,540,495 positions,   4,014 trips ->  612,420 points (R = 86.5 %),     413 s elapsed, memory 10.2 GiB so far


2017-11: block of 4,529,720 positions,   3,553 trips ->  452,224 points (R = 90.0 %),     500 s elapsed, memory 10.3 GiB so far


... (18 outputs left out)

2017-12: block of 4,497,788 positions,   6,815 trips ->  445,402 points (R = 90.1 %),    2056 s elapsed, memory 10.7 GiB so far


2017-12: block of 4,511,316 positions,   4,258 trips ->  145,685 points (R = 96.8 %),    2139 s elapsed, memory 10.7 GiB so far


2017-12: block of 1,383,398 positions,   1,379 trips ->   36,643 points (R = 97.4 %),    2197 s elapsed, memory 10.7 GiB so far



115,022,399 positions read (military group left out), split into 292,382 trips at gaps of over 30 min; dropped 159,724 tiny trips of fewer than 10 positions (420,696 positions, 0.37% of the data)
dropped the groups ('Military',): 10,000 positions of 4 vessels (as in the paper)
kept 132,658 trips, 114,601,703 positions
compressed to 8,123,613 points; whole run 2197 s on -1 job(s); peak memory of this process and its workers 10.7 GiB


### The summary table

`summary.csv` has one row per month, per vessel group, per vessel type and one for all of them (`level`), with the number of
vessels, trips, points in and out (and stay points out), the compression ratio and, for the months and `ALL`, the seconds (the months': wall time of reading, trips, compression, checks and writing; the row `ALL`: compression only). The row `ALL` also carries the timings of the steps, the number of jobs and the memory peak.

In [7]:
tab = pd.concat(tabs).groupby(level=[0, 1]).sum()
vk = pd.concat(vessel_keys).drop_duplicates()
tab["vessels"] = vk.groupby(["vessel_group", "vessel_type"]).mmsi.nunique()
grp = tab.groupby(level=0).sum()
grp["vessels"] = vk.groupby("vessel_group").mmsi.nunique()
cols = ["vessels", "trips", "points_in", "points_out", "stay_points_out"]
rows = [dict(level="type", name=t, vessel_group=g, **tab.loc[(g, t), cols].to_dict()) for g, t in tab.index]
rows += [dict(level="group", name=g, vessel_group=g, **grp.loc[g, cols].to_dict()) for g in grp.index]
rows += month_rows
rows.append(dict(level="ALL", name="ALL", vessels=vk.mmsi.nunique(), trips=N_TRIPS, points_in=N,
                 points_out=int(tab.points_out.sum()), stay_points_out=int(tab.stay_points_out.sum()),
                 seconds=COMPRESS_S))
summary = pd.DataFrame(rows)
for c in ["vessels", "trips", "points_in", "points_out", "stay_points_out"]:
    summary[c] = summary[c].fillna(0).astype("int64")
summary["compression_ratio_pct"] = [compression_ratio(o, i) for o, i in zip(summary.points_out, summary.points_in)]
summary["n_jobs"] = N_JOBS
for key, val in dict(read_seconds=READ_S, split_trips_seconds=SPLIT_S, compress_seconds=COMPRESS_S, check_seconds=CHECK_S,
                     write_seconds=WRITE_S, run_seconds=RUN_S, peak_memory_gib=PEAK_GIB).items():
    summary[key] = np.where(summary.level == "ALL", val, np.nan)
summary = summary[["level", "name", "vessel_group", "vessels", "trips", "points_in", "points_out", "stay_points_out",
                   "compression_ratio_pct", "seconds", "n_jobs", "read_seconds", "split_trips_seconds",
                   "compress_seconds", "check_seconds", "write_seconds", "run_seconds", "peak_memory_gib"]]
summary.to_csv(OUT / "summary.csv", index=False)
for f in sorted(OUT.iterdir()):
    print(f"{f.name:34s} {f.stat().st_size / 1e6:10.2f} MB")

compressed_ais_2017-11.parquet         165.05 MB
compressed_ais_2017-12.parquet         155.24 MB
summary.csv                              0.00 MB
trips_ais_2017-11.parquet             1288.81 MB
trips_ais_2017-12.parquet             1166.58 MB


## 7 · Results

**Compression ratio** R is the percentage of points removed (Eq. 26 of the paper):
`R = (1 - points out / points in) * 100`. First overall and by vessel group; then
the vessel groups and types are compared in charts. Different vessels compress very differently:
a vessel that sails a straight line or sits at a berth gets a very high ratio, one that
turns and changes speed all the time keeps many points.

**Compared with the paper.** The paper reports about 80 % compression for 2stage-CPLS on the eastern US coast, November and December 2017 (abstract and section V-A,
where the ratio is shown against the offset distance for 500 random trajectories). The number printed below is **not a reproduction** of that result: it is
the ratio of this implementation on this preprocessing of the same months. What differs: the study area is the author's polygon, not the paper's
(about twice as many positions); the vessels are cleaned by a stricter rule (ship MMSIs only, ranges, jumps) and median-filtered, the paper used a moving average; trips here
are cut at 30-minute gaps and short ones dropped, the paper's trips are those of MovingPandas and the 500 test trajectories were drawn at random from
them; and the ratio here is over **all** trips of the two months, with the shoreline of the study area, ε = 1 km. It is a figure to put beside the paper's, not to match to it.

In [8]:
by_level = lambda lv: summary[summary.level == lv]
allrow = by_level("ALL").iloc[0]
R_ALL = allrow.compression_ratio_pct
print(f"overall: {N:,} -> {int(allrow.points_out):,} points, R = {R_ALL:.2f} % "
      f"({N / allrow.points_out:.1f} points in per point kept), {int(allrow.trips):,} trips, {int(allrow.vessels):,} vessels, "
      f"{int(allrow.stay_points_out):,} stay points kept")
print(f"the paper reports about 80 %: this run is {R_ALL - 80:+.1f} percentage points from it (not a reproduction: see the text above)")
show = summary[summary.level.isin(["ALL", "month", "group"])].drop(columns=[c for c in summary if c.endswith("_seconds") or c in ("n_jobs", "peak_memory_gib")])
show = show.assign(compression_ratio_pct=show.compression_ratio_pct.round(2), seconds=show.seconds.round(1)).drop(columns="vessel_group")
show.set_index(["level", "name"])

overall: 114,601,703 -> 8,123,613 points, R = 92.91 % (14.1 points in per point kept), 132,658 trips, 12,057 vessels, 702,237 stay points kept
the paper reports about 80 %: this run is +12.9 percentage points from it (not a reproduction: see the text above)


vessels   trips  points_in  points_out  stay_points_out  \
level name                                                                 
group Cargo         1854    8720   10416463      273339            20374   
      Fishing        810   14645    7403181      440064            25564   
      Other         3820   47067   54787799     4118822           396229   
      Passenger      689   15559   10627836     2544747           142576   
      Sailing       4062   41848   27114299      583413           101184   
      Tanker         787    4215    2734951       65364             6968   
      Tug             35     604    1517174       97864             9342   
month 2017-11      10045   71091   59544706     4181457           364950   
      2017-12       8524   61567   55056997     3942156           337287   
ALL   ALL          12057  132658  114601703     8123613           702237   

                 compression_ratio_pct  seconds  
level name                                       
group Cargo                      97.38      NaN  
      Fishing                    94.06      NaN  
      Other                      92.48      NaN  
      Passenger                  76.06      NaN  
      Sailing                    97.85      NaN  
      Tanker                     97.61      NaN  
      Tug                        93.55      NaN  
month 2017-11                    92.98   1165.0  
      2017-12                    92.84   1031.7  
ALL   ALL                        92.91   1497.0

In [9]:
def ratio_chart(df, label, title, height):
    df = df.sort_values("compression_ratio_pct").copy()
    df["label"] = [f"{r:.1f} %  ({n:,} pts)" for r, n in zip(df.compression_ratio_pct, df.points_in)]
    df["pos"] = df.compression_ratio_pct + 1.0
    bars = df.hvplot.barh(x="name", y="compression_ratio_pct", color="#3b6fb6", xlabel="",
                          ylabel="compression ratio R (% of points removed)",
                          hover_cols=["points_in", "points_out", "vessels", "trips"], ylim=(0, 112),
                          width=800, height=height, title=title)
    labels = hv.Labels(df, ["name", "pos"], "label").opts(text_align="left", text_font_size="8pt")
    return bars * labels * hv.VLine(R_ALL).opts(color="#cf222e", line_width=1.5)


groups = by_level("group")
ratio_chart(groups, "name", "2stage-CPLS: compression ratio by vessel group (red line: all)", int(34 * len(groups) + 110))

:Overlay
   .Bars.I   :Bars   [name]   (compression_ratio_pct,points_in,points_out,vessels,trips)
   .Labels.I :Labels   [name,pos]   (label)
   .VLine.I  :VLine   [x,y]

In [10]:
types = by_level("type")
types = types[types.points_in >= max(1_000, 0.0005 * N)]          # types with enough positions to say something
ratio_chart(types, "name", f"2stage-CPLS: compression ratio by vessel type ({len(types)} types with at least "
            f"{int(types.points_in.min()):,} positions)", int(22 * len(types) + 110))

:Overlay
   .Bars.I   :Bars   [name]   (compression_ratio_pct,points_in,points_out,vessels,trips)
   .Labels.I :Labels   [name,pos]   (label)
   .VLine.I  :VLine   [x,y]

### Vessels before and after

Three vessels, one from each of the three vessel groups with most positions (the vessel with
the most positions up to 5,000, with a stay, and a track between 0.1 and 3 degrees across (about 10 to 300 km), so that the map is readable and the vessel actually moved): the clean positions in grey (trip by trip), the compressed points joined by
their segments in blue, the **stays** (the two centroid points of every stay) as orange squares,
the NOAA shoreline in brown. The left panel is the original, the right panel the compressed track. Both panels are interactive: zoom and pan, hover a point for its time and speed.
The tracks of these vessels are read back from the saved Parquet files.

In [11]:
LAND, INK, ORANGE, BLUE = "#8a6d3b", "#0d1117", "#e16f24", "#1f5fa8"
coast_ll = coast.to_crs(4326)
coast_ll["geometry"] = coast_ll.geometry.simplify(0.0005 if len(coast) < 5_000 else 0.005)     # ~50 m / ~500 m: plenty for a map
vessels = (pd.concat(vessel_info).groupby("mmsi").agg(n=("n", "sum"), lon0=("lon0", "min"), lon1=("lon1", "max"),
           lat0=("lat0", "min"), lat1=("lat1", "max"), vessel_group=("vessel_group", "first"),
           vessel_type=("vessel_type", "first"), stays=("stays", "sum")))


def pick_vessels(k=3, cap=5_000):
    """From each of the k largest vessel groups: the vessel with most positions (up to the cap) that has a stay
    and a compact track."""
    ext = np.maximum(vessels.lon1 - vessels.lon0, vessels.lat1 - vessels.lat0)
    compact = (ext < 3) & (ext > 0.1)                     # neither a river of a track nor a vessel that never left its berth
    out = []
    for group in groups.sort_values("points_in", ascending=False).name.head(k):
        for pool in (vessels[compact & (vessels.stays > 0)], vessels[compact], vessels):
            pool = pool[(pool.vessel_group == group) & (pool.n <= cap)]
            if len(pool):
                out.append((group, int(pool.n.idxmax()))); break
    return out


def read_vessels(prefix, mmsis, columns):
    parts = [pd.read_parquet(f, columns=columns, filters=[("mmsi", "in", list(mmsis))])
             for f in sorted(OUT.glob(prefix + "_*.parquet"))]
    return pd.concat(parts, ignore_index=True)


def basemap(lon0, lon1, lat0, lat1):
    """Map tiles with the shoreline lines inside the box, drawn first (below the tracks)."""
    base = gv.tile_sources.CartoLight()
    box = gpd.GeoDataFrame(geometry=[shapely.box(lon0, lat0, lon1, lat1)], crs=4326)
    part = gpd.clip(coast_ll, box)
    if len(part):
        base = base * part.hvplot.paths(geo=True, color=LAND, line_width=1.5, hover=False)
    return base


def trip_lines(df, **opts):
    """One line per trip (no chord across the gap between two trips), as a geographic path layer."""
    g = gpd.GeoDataFrame({"trip_id": df.trip_id.unique()},
                         geometry=[shapely.LineString(np.c_[t.lon, t.lat]) for _, t in df.groupby("trip_id", sort=False)],
                         crs=4326)
    return g.hvplot.paths(geo=True, hover=False, **opts)


def with_time(df):
    df = df[["trip_id", "lon", "lat", "unix_time", "speed_over_ground"]].copy()
    df["time"] = pd.to_datetime(df.unix_time, unit="s")
    return df


HOVER = ["time", "speed_over_ground"]
chosen = pick_vessels()
mm = [m for _, m in chosen]
cols = ["mmsi", "trip_id", "lon", "lat", "unix_time", "speed_over_ground"]
orig, comp = read_vessels("trips_ais", mm, cols), read_vessels("compressed_ais", mm, cols + ["stay"])
panels = []
for group, mmsi in chosen:
    before = with_time(orig[orig.mmsi == mmsi])
    aft = comp[comp.mmsi == mmsi]
    after, st = with_time(aft), with_time(aft[aft.stay])
    pad = max(0.02, 0.12 * max(np.ptp(before.lon) * 0.7, np.ptp(before.lat)))
    base = basemap(before.lon.min() - pad, before.lon.max() + pad, before.lat.min() - pad, before.lat.max() + pad)
    m0 = base * trip_lines(before, color="#777777", line_width=1)
    m0 = m0 * before.hvplot.points(x="lon", y="lat", geo=True, color="#555555", size=3, hover_cols=HOVER)
    m1 = base * trip_lines(after, color=BLUE, line_width=2)
    m1 = m1 * after.hvplot.points(x="lon", y="lat", geo=True, color=INK, size=6, hover_cols=HOVER)
    if len(st):
        m1 = m1 * st.hvplot.points(x="lon", y="lat", geo=True, color=ORANGE, marker="square", size=60,
                                   line_color=INK, hover_cols=["time"], label="stay (centroid)")
    kind = vessels.vessel_type[mmsi]
    m0 = m0.opts(width=520, height=440, title=f"{group} ({kind}), MMSI {mmsi}: original, {len(before):,} points")
    m1 = m1.opts(width=520, height=440,
                 title=f"compressed, {len(after):,} points (R = {compression_ratio(len(after), len(before)):.1f} %)")
    panels += [m0, m1]
hv.Layout(panels).cols(2).opts(shared_axes=False)

:Layout
   .Overlay.I   :Overlay
      .WMTS.I   :WMTS   [Longitude,Latitude]
      .Path.I   :Path   [Longitude,Latitude]
      .Path.II  :Path   [Longitude,Latitude]
      .Points.I :Points   [lon,lat]   (time,speed_over_ground)
   .Overlay.II  :Overlay
      .WMTS.I                                                  :WMTS   [Longitude,Latitude]
      .Path.I                                                  :Path   [Longitude,Latitude]
      .Path.II                                                 :Path   [Longitude,Latitude]
      .Points.I                                                :Points   [lon,lat]   (time,speed_over_ground)
      .Points.Stay_left_parenthesis_centroid_right_parenthesis :Points   [lon,lat]   (time)
   .Overlay.III :Overlay
      .WMTS.I   :WMTS   [Longitude,Latitude]
      .Path.I   :Path   [Longitude,Latitude]
      .Path.II  :Path   [Longitude,Latitude]
      .Points.I :Points   [lon,lat]   (time,speed_over_ground)
   .Overlay.IV  :Overlay
      .WMTS.I                                                  :WMTS   [Longitude,Latitude]
      .Path.I                                                  :Path   [Longitude,Latitude]
      .Path.II                                                 :Path   [Longitude,Latitude]
      .Points.I                                                :Points   [lon,lat]   (time,speed_over_ground)
      .Points.Stay_left_parenthesis_centroid_right_parenthesis :Points   [lon,lat]   (time)
   .Overlay.V   :Overlay
      .WMTS.I   :WMTS   [Longitude,Latitude]
      .Path.I   :Path   [Longitude,Latitude]
      .Path.II  :Path   [Longitude,Latitude]
      .Points.I :Points   [lon,lat]   (time,speed_over_ground)
   .Overlay.VI  :Overlay
      .WMTS.I                                                  :WMTS   [Longitude,Latitude]
      .Path.I                                                  :Path   [Longitude,Latitude]
      .Path.II                                                 :Path   [Longitude,Latitude]
      .Points.I                                                :Points   [lon,lat]   (time,speed_over_ground)
      .Points.Stay_left_parenthesis_centroid_right_parenthesis :Points   [lon,lat]   (time)

### Stays

A stay is replaced by two points at its centroid, the arrival and the departure, with their own times, so the
position stays in one place while the time passes. The map shows the compressed points of the
dataset (a random sample of at most 6,000 drawn, with up to 2,000 of them stay points, to keep the notebook light): ordinary points in blue, stay points in orange, over the shoreline.
Stays sit where vessels wait: in ports, at moorings and at anchor, along the coast. The line below counts
the stay points kept (each stay of two or more points leaves two flagged points, a one-point stay one).

In [12]:
cols = ["trip_id", "mmsi", "lon", "lat", "vessel_group", "speed_over_ground", "stay"]
allc = pd.concat([pd.read_parquet(f, columns=cols) for f in sorted(OUT.glob("compressed_ais_*.parquet"))], ignore_index=True)
print(f"{int(allc.stay.sum()):,} stay points kept; "
      f"{allc[allc.stay].trip_id.nunique():,} trips contain a stay; {allc[allc.stay].mmsi.nunique():,} vessels")
st_draw = allc[allc.stay].sample(min(2_000, int(allc.stay.sum())), random_state=0)
ns_all = allc[~allc.stay]
ns_draw = ns_all.sample(min(6_000 - len(st_draw), len(ns_all)), random_state=0)
n_kept = len(allc)
del allc, ns_all
coast_view = coast_ll.copy()
if len(coast) >= 5_000:                                        # the whole east coast: only the long lines, coarsely
    coast_view = coast_view[coast.length.to_numpy() > 10_000]
coast_view["geometry"] = coast_view.geometry.simplify(0.002 if len(coast) < 5_000 else 0.03)      # ~200 m / ~3 km: the whole area at once
land_all = gv.tile_sources.CartoLight() * coast_view.hvplot.paths(geo=True, color=LAND, line_width=0.8, hover=False)
hovers = ["vessel_group", "speed_over_ground"]
(land_all
 * pd.DataFrame(ns_draw[["lon", "lat"] + hovers]).hvplot.points(x="lon", y="lat", geo=True, color=BLUE, size=3, alpha=0.5,
                                                                 hover_cols=hovers, label="compressed points")
 * pd.DataFrame(st_draw[["lon", "lat"] + hovers]).hvplot.points(x="lon", y="lat", geo=True, color=ORANGE, size=7,
                                                                 hover_cols=hovers, label="stay points (centroids)")
 ).opts(width=900, height=700, legend_position="top_right",
        title=f"Compressed points and stays (a sample of {len(ns_draw) + len(st_draw):,} of {n_kept:,})")

702,237 stay points kept; 106,214 trips contain a stay; 10,622 vessels


:Overlay
   .WMTS.I                                                          :WMTS   [Longitude,Latitude]
   .Path.I                                                          :Path   [Longitude,Latitude]
   .Points.Compressed_points                                        :Points   [lon,lat]   (vessel_group,speed_over_ground)
   .Points.Stay_points_left_parenthesis_centroids_right_parenthesis :Points   [lon,lat]   (vessel_group,speed_over_ground)

## 8 · Checks, timings and files

**The checks of section 5**, summed over all blocks. Against the collapsed track there should be no new crossing of the shoreline, as
guaranteed, and recomputing the selection with the array function should reproduce the saved rows exactly (the
notebook stops with an error otherwise). Against the raw track, the first line counts what the
move of stays to their centroid costs, if anything: the documented price of replacing the
jitter of a stay by its centroid. On the full data these are measured on random shares of the trips (`CHECK_RAW`, `CHECK_COLLAPSED`).

In [13]:
c = check_totals
print(f"against the raw track: {c['raw_new']} new shoreline crossing(s) in {c['raw_trips_with_new']} of {c['raw_trips']:,} trips "
      f"checked ({c['raw_trips'] / N_TRIPS:.0%} of the trips; {c['raw_new_at_stay']} of them at a segment end that is a stay point)")
print(f"against the collapsed track: {c['col_new']} new shoreline crossing(s) in {c['col_trips']:,} trips checked "
      f"({c['col_trips'] / N_TRIPS:.0%} of the trips); the recomputed selection equals the saved rows in {c['col_same']:,} of them")
assert c["col_new"] == 0 and c["col_same"] == c["col_trips"], "2stage-CPLS guarantee violated"

against the raw track: 69 new shoreline crossing(s) in 63 of 6,526 trips checked (5% of the trips; 69 of them at a segment end that is a stay point)
against the collapsed track: 0 new shoreline crossing(s) in 1,362 trips checked (1% of the trips); the recomputed selection equals the saved rows in 1,362 of them


In [14]:
print(f"barriers {LOAD_S:.1f} s | read {READ_S:.1f} s | split_trips {SPLIT_S:.1f} s | compress {COMPRESS_S:.1f} s | "
      f"checks {CHECK_S:.1f} s | write {WRITE_S:.1f} s | whole run {RUN_S:.1f} s")
print(f"N_JOBS = {N_JOBS} ({os.cpu_count()} logical cores); {N / COMPRESS_S:,.0f} positions per second compressed; "
      f"peak memory {PEAK_GIB:.1f} GiB")
files = sorted(OUT.iterdir())
pd.DataFrame({"file": [f.name for f in files], "MB": [round(f.stat().st_size / 1e6, 2) for f in files]}).set_index("file")

barriers 0.1 s | read 11.8 s | split_trips 152.5 s | compress 1497.0 s | checks 433.8 s | write 58.5 s | whole run 2196.7 s
N_JOBS = -1 (16 logical cores); 76,554 positions per second compressed; peak memory 10.7 GiB


,MB
file,
compressed_ais_2017-11.parquet,165.05
compressed_ais_2017-12.parquet,155.24
summary.csv,0.00
trips_ais_2017-11.parquet,1288.81
trips_ais_2017-12.parquet,1166.58


## Notes

* The compressed data is **a row subset** of the `trips_ais_*.parquet` files (same `trip_id`, same
  `unix_time`, same measured columns); the only changed values are the position of the stay
  points (`x`, `y`, `lon`, `lat`), which sit at their stay's centroid.
* A trip needs at least 10 positions to be kept; both outputs use the same trips. `trip_id` is unique over both months; trips are cut at the month boundary.
* The speed threshold is in metres per second for UNIX-second times: 1.54 m/s is 3 knots.
* Everything generated here is written under `results/` and is not committed; the committed
  copy of this notebook is the executed run on the sample. The run on the full data is saved as
  `notebooks/full_runs/03_two_stage_cpls_compression.ipynb`.

**Citing.** MovingPandas: Graser, A. (2019), GI_Forum 7(1), 54–68,
[doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54). CPLS and
2stage-CPLS: Mehri, S., Alesheikh, A. A., Basiri, A. (2021), IEEE Access 9, 45600–45613,
[doi:10.1109/ACCESS.2021.3066463](https://doi.org/10.1109/ACCESS.2021.3066463). Stay points: Li, Q.
et al. (2008), ACM SIGSPATIAL GIS; Yu, Y. (2015). Two-stage compression: de Vries, G. K. D. and van
Someren, M. (2012), Expert Systems with Applications 39(18), 13426–13439. Data: AIS broadcasts of the U.S. Coast Guard,
distributed by MarineCadastre.gov (BOEM and NOAA), https://marinecadastre.gov/ais/; shoreline: NOAA Medium Resolution Shoreline
(NOAA Office of Coast Survey / National Geodetic Survey).